# Strict vs. lenient evaluation of out-of-fold predictions

Reads the multi-column BIO files `oof.gold.bio.tsv` and `oof.pred.bio.tsv` (written by `evaluate_cv.py`), rebuilds the spans and computes P, R and F1 under three matching criteria:

| Mode | A prediction is correct if… |
|---|---|
| **strict** | start, end and label match exactly |
| **lenient** | it overlaps a gold span with the same label by at least one token |
| **partial** | as lenient, but a non-exact match counts 0.5 (SemEval-2013) |

Matching is one-to-one: each gold span can be matched by at most one prediction and vice versa. The notebook then breaks errors down by type (boundary, label, missed, spurious).

Since the OOF files carry no fold information, all scores are pooled over folds; uncertainty is estimated with a bootstrap over sentences instead of the across-fold standard deviation.

## Setup

In [ ]:
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

GOLD_PATH = Path("oof.gold.bio.tsv")
PRED_PATH = Path("oof.pred.bio.tsv")
CV_SUMMARY_PATH = Path("cv_summary.tsv")   # optional, only used for the consistency check
OUT_DIR = Path("lenient_results")
OUT_DIR.mkdir(exist_ok=True)

N_BOOTSTRAP = 1000
SEED = 42

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

## Reading the BIO files

One token per line, one column per frame, blank lines between sentences. A cell may hold several tags separated by `|` when spans of the same frame overlap on that token (e.g. `B-DESCR_Neutral|B-DESCR_Positive`, or `I-FOOD_LU|B-FOOD_LU` for a nested span).

Malformed tags (`B_CURE_LU`, `o`, `BO`, empty cells, …) are treated as `O`, which is also how `cv_summary.tsv` was produced; they are listed below.

In [ ]:
VALID_TAG = re.compile(r"^[BI]-\S+$")

def read_bio(path):
    """Return (frames, sentences); a sentence is a list of rows [token, tag_frame1, ...]."""
    sents, cur = [], []
    with open(path, encoding="utf-8") as f:
        frames = f.readline().rstrip("\n").split("\t")[1:]
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                if cur:
                    sents.append(cur)
                    cur = []
                continue
            cur.append(line.split("\t"))
    if cur:
        sents.append(cur)
    return frames, sents


def malformed_tags(frames, sents, name):
    rows = []
    for si, s in enumerate(sents):
        for ti, row in enumerate(s):
            for fi, fr in enumerate(frames):
                for t in row[fi + 1].split("|"):
                    if t != "O" and not VALID_TAG.match(t):
                        rows.append(dict(file=name, sent=si, pos=ti, token=row[0], frame=fr, tag=repr(t)))
    return pd.DataFrame(rows)


FRAMES, gold_sents = read_bio(GOLD_PATH)
pred_frames, pred_sents = read_bio(PRED_PATH)

assert FRAMES == pred_frames, "different frame columns"
assert len(gold_sents) == len(pred_sents), "different number of sentences"
for g, p in zip(gold_sents, pred_sents):
    assert [r[0] for r in g] == [r[0] for r in p], "tokens not aligned"

print(f"Frames: {FRAMES}")
print(f"Sentences: {len(gold_sents)}   Tokens: {sum(len(s) for s in gold_sents)}")

malformed = pd.concat([malformed_tags(FRAMES, gold_sents, "gold"),
                       malformed_tags(FRAMES, pred_sents, "pred")], ignore_index=True)
print(f"\nMalformed tags (read as O): {len(malformed)}")
malformed

## From BIO tags to spans

Tags are read token by token, separately for each label: `B-X` opens a span, `I-X` continues an open span of type X (the longest one if several are open), an orphan `I-X` opens a new span (as in `conlleval`), and a span closes when the next token does not continue it. Counting tags per label handles nested spans of the same type, e.g. *Crab Apple* and *Apple*, both `FOOD_LU`.

A span is `(sentence, frame, label, start, end)`, with `end` exclusive.

In [ ]:
def normalise(tag):
    tag = tag.strip()
    return tag if VALID_TAG.match(tag) else "O"


def extract_spans(frames, sents):
    spans = []
    for si, s in enumerate(sents):
        rows = s + [[None] + ["O"] * len(frames)]      # sentinel row closes open spans
        for fi, fr in enumerate(frames):
            open_ = defaultdict(list)                   # label -> start positions, oldest first
            for ti, row in enumerate(rows):
                n_I, n_B = Counter(), Counter()
                for t in (normalise(x) for x in row[fi + 1].split("|")):
                    if t == "O":
                        continue
                    pref, lab = t.split("-", 1)
                    (n_I if pref == "I" else n_B)[lab] += 1
                still_open = defaultdict(list)
                for lab, starts in open_.items():
                    k = min(n_I[lab], len(starts))
                    still_open[lab] += starts[:k]               # continued
                    for st in starts[k:]:                       # closed
                        spans.append((si, fr, lab, st, ti))
                    n_I[lab] -= k
                for lab, c in n_I.items():                      # orphan I-
                    still_open[lab] += [ti] * c
                for lab, c in n_B.items():                      # B-
                    still_open[lab] += [ti] * c
                open_ = still_open
    df = pd.DataFrame(spans, columns=["sent", "frame", "label", "start", "end"])
    df["text"] = [" ".join(r[0] for r in sents[s][a:b]) for s, a, b in zip(df.sent, df.start, df.end)]
    return df


gold = extract_spans(FRAMES, gold_sents)
pred = extract_spans(FRAMES, pred_sents)

# label order: by frame, then by first occurrence in the gold
LABELS = list(dict.fromkeys(gold.sort_values(["frame"], key=lambda s: s.map(FRAMES.index), kind="stable")["label"]))
LABEL2FRAME = dict(zip(gold.label, gold.frame))
print(f"Gold spans: {len(gold)}   Predicted spans: {len(pred)}   Labels: {len(LABELS)}")
gold.head()

## Matching gold and predicted spans

Within each *(sentence, frame, label)* group, identical spans are matched first; the remaining ones are matched by decreasing overlap, without reusing a span. This gives, per group, `n_gold`, `n_pred`, `exact` and `partial` matches, from which all scores follow:

- **strict**: TP = exact
- **lenient**: TP = exact + partial
- **partial**: TP = exact + 0.5 × partial

with P = TP / n_pred and R = TP / n_gold.

In [ ]:
def overlap(a, b):
    return max(0, min(a[1], b[1]) - max(a[0], b[0]))


def match(g_spans, p_spans):
    """g_spans, p_spans: lists of (start, end). Returns (n_exact, n_partial, partial_pairs)."""
    g_free, p_free = list(range(len(g_spans))), list(range(len(p_spans)))
    exact = 0
    for i in list(g_free):
        for j in p_free:
            if g_spans[i] == p_spans[j]:
                g_free.remove(i); p_free.remove(j); exact += 1
                break
    cand = sorted(((overlap(g_spans[i], p_spans[j]), i, j)
                   for i in g_free for j in p_free if overlap(g_spans[i], p_spans[j]) > 0),
                  reverse=True)
    pairs = []
    for _, i, j in cand:
        if i in g_free and j in p_free:
            g_free.remove(i); p_free.remove(j)
            pairs.append((g_spans[i], p_spans[j]))
    return exact, len(pairs), pairs


keys = ["sent", "frame", "label"]
g_grp = gold.groupby(keys)[["start", "end"]].apply(lambda d: list(zip(d.start, d.end)))
p_grp = pred.groupby(keys)[["start", "end"]].apply(lambda d: list(zip(d.start, d.end)))

rows, boundary_examples = [], []
for k in g_grp.index.union(p_grp.index):
    gs, ps = g_grp.get(k, []), p_grp.get(k, [])
    e, p, pairs = match(gs, ps)
    rows.append((*k, len(gs), len(ps), e, p))
    for (ga, gb), (pa, pb) in pairs:
        toks = [r[0] for r in gold_sents[k[0]]]
        boundary_examples.append(dict(sent=k[0], frame=k[1], label=k[2],
                                      gold=" ".join(toks[ga:gb]), pred=" ".join(toks[pa:pb]),
                                      type=("too long" if pa <= ga and pb >= gb else
                                            "too short" if pa >= ga and pb <= gb else "shifted")))

counts = pd.DataFrame(rows, columns=keys + ["n_gold", "n_pred", "exact", "partial"])
boundary_examples = pd.DataFrame(boundary_examples)
counts.head()

### Consistency check with `cv_summary.tsv`

Strict counts should match the `tp`/`fp` of the summary file; if they do, span reconstruction is identical to the original evaluation and lenient scores are directly comparable.

In [ ]:
if CV_SUMMARY_PATH.exists():
    cv = pd.read_csv(CV_SUMMARY_PATH, sep="\t")
    cv = cv[~cv["label"].str.startswith("<")].set_index("label")
    tot = counts.groupby("label")[["n_gold", "n_pred", "exact"]].sum()
    chk = pd.DataFrame({
        "gold (here)": tot.n_gold, "gold (cv)": cv.gold_spans,
        "tp (here)": tot.exact, "tp (cv)": cv.tp,
        "fp (here)": tot.n_pred - tot.exact, "fp (cv)": cv.fp,
    }).reindex(LABELS)
    diff = ((chk["gold (here)"] != chk["gold (cv)"]) | (chk["tp (here)"] != chk["tp (cv)"])
            | (chk["fp (here)"] != chk["fp (cv)"]))
    print("Strict counts match cv_summary.tsv" if not diff.any()
          else f"Differences on {diff.sum()} labels:")
    display(chk[diff] if diff.any() else chk)
else:
    print("cv_summary.tsv not found: check skipped")

## Strict, partial and lenient scores

Per label, per frame (micro over the frame's labels) and overall (micro, and macro as the unweighted mean of label F1). The summary is also saved in the same layout as `cv_summary.tsv`.

In [ ]:
MODES = ["strict", "partial", "lenient"]


def prf(n_gold, n_pred, exact, partial, mode):
    tp = exact + {"strict": 0, "partial": 0.5, "lenient": 1}[mode] * partial
    p = np.divide(tp, n_pred, out=np.zeros_like(tp, dtype=float), where=np.asarray(n_pred) > 0)
    r = np.divide(tp, n_gold, out=np.zeros_like(tp, dtype=float), where=np.asarray(n_gold) > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros_like(p, dtype=float), where=(p + r) > 0)
    return p, r, f


def add_scores(agg):
    out = agg.copy()
    for m in MODES:
        p, r, f = prf(agg.n_gold.values.astype(float), agg.n_pred.values.astype(float),
                      agg.exact.values.astype(float), agg.partial.values.astype(float), m)
        out[f"P_{m}"], out[f"R_{m}"], out[f"F1_{m}"] = p, r, f
    return out


cols = ["n_gold", "n_pred", "exact", "partial"]
per_label = add_scores(counts.groupby("label")[cols].sum()).reindex(LABELS)
per_label.insert(0, "frame", per_label.index.map(LABEL2FRAME))

per_frame = add_scores(counts.groupby("frame")[cols].sum()).reindex(FRAMES)
overall = add_scores(counts[cols].sum().to_frame("<micro>").T)
macro = pd.DataFrame({f"F1_{m}": [per_label[f"F1_{m}"].mean()] for m in MODES}, index=["<macro>"])

summary = pd.concat([
    per_label.reset_index(),
    per_frame.reset_index().rename(columns={"index": "frame"}).assign(label="<micro>"),
    overall.assign(frame="<ALL>").reset_index(names="label"),
    macro.assign(frame="<ALL>").reset_index(names="label"),
], ignore_index=True)
summary = summary[["frame", "label"] + cols + [f"{x}_{m}" for m in MODES for x in ["P", "R", "F1"]]]
summary.to_csv(OUT_DIR / "cv_summary_lenient.tsv", sep="\t", index=False, float_format="%.4f")

view = ["frame", "label", "n_gold", "F1_strict", "F1_partial", "F1_lenient"]
summary[view].assign(gain=summary.F1_lenient - summary.F1_strict).round(3)

### Bootstrap confidence intervals

Sentences (not spans, which are not independent within a sentence) are resampled with replacement; the 95% interval is given by the 2.5th and 97.5th percentiles.

In [ ]:
rng = np.random.default_rng(SEED)
n_sents = len(gold_sents)

# sentence x label matrix for each count: each bootstrap sample is a weighted sum
mat = {c: counts.pivot_table(index="sent", columns="label", values=c, aggfunc="sum", fill_value=0)
             .reindex(index=range(n_sents), columns=LABELS, fill_value=0).values.astype(float)
       for c in cols}

weights = rng.multinomial(n_sents, np.full(n_sents, 1 / n_sents), size=N_BOOTSTRAP)   # (B, sentences)
boot = {c: weights @ mat[c] for c in cols}                                            # (B, labels)

ci = {}
for m in MODES:
    _, _, f_lab = prf(boot["n_gold"], boot["n_pred"], boot["exact"], boot["partial"], m)
    _, _, f_mic = prf(*(boot[c].sum(1) for c in cols), m)
    f_mac = f_lab.mean(1)
    ci[m] = pd.DataFrame(
        np.vstack([np.percentile(f_lab, [2.5, 97.5], axis=0).T,
                   np.percentile(f_mic, [2.5, 97.5])[None],
                   np.percentile(f_mac, [2.5, 97.5])[None]]),
        index=LABELS + ["<micro>", "<macro>"], columns=["lo", "hi"])

pd.concat({m: ci[m] for m in MODES}, axis=1).round(3)

## Error breakdown

Lenient scores show how much is lost to boundaries; this section shows where errors come from. Each **gold** span is assigned to the closest prediction in the same frame:

| Category | Meaning |
|---|---|
| exact | identical span and label |
| boundary | same label, different boundaries |
| label | same span, different label within the frame |
| label + boundary | overlap with a different label |
| missed | no overlapping prediction in the frame |

This assignment is not one-to-one, so it is meant for diagnosis rather than scoring. It also yields a confusion matrix between labels of the same frame. Predictions that overlap no gold span of their frame are counted as spurious.

In [ ]:
CATEGORIES = ["exact", "boundary", "label", "label + boundary", "missed"]

pred_idx = defaultdict(list)
for r in pred.itertuples():
    pred_idx[(r.sent, r.frame)].append((r.start, r.end, r.label))

cat, lab_pred = [], []
for r in gold.itertuples():
    best, best_lab = 4, None
    for a, b, l in pred_idx[(r.sent, r.frame)]:
        if overlap((r.start, r.end), (a, b)) == 0:
            continue
        same_span, same_lab = (a, b) == (r.start, r.end), l == r.label
        c = 0 if same_span and same_lab else 1 if same_lab else 2 if same_span else 3
        if c < best:
            best, best_lab = c, l
    cat.append(CATEGORIES[best])
    lab_pred.append(best_lab if best_lab else "(none)")
gold["category"], gold["label_pred"] = cat, lab_pred

breakdown = pd.crosstab(gold.label, gold.category).reindex(index=LABELS, columns=CATEGORIES, fill_value=0)

gold_idx = defaultdict(list)
for r in gold.itertuples():
    gold_idx[(r.sent, r.frame)].append((r.start, r.end))
pred["spurious"] = [not any(overlap((r.start, r.end), g) for g in gold_idx[(r.sent, r.frame)])
                    for r in pred.itertuples()]
breakdown["spurious (pred)"] = pred.groupby("label")["spurious"].sum().reindex(LABELS).fillna(0).astype(int)
breakdown

## Plots

In [ ]:
palette = plt.get_cmap("tab10")
FCOLOR = {f: palette(i) for i, f in enumerate(FRAMES)}
MCOLOR = {"strict": "#4C72B0", "partial": "#8C8C8C", "lenient": "#DD8452"}
short = lambda s: s.replace("COOKING_CREATION_", "CC_")
frame_legend = [Patch(color=FCOLOR[f], label=f) for f in FRAMES]

def save(fig, name):
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

### Strict vs. lenient F1 per label
Each line goes from strict (blue) to lenient (orange) F1, with partial in grey and 95% bootstrap intervals; the longer the line, the larger the share of boundary errors.

In [ ]:
d = per_label.iloc[::-1]
y = np.arange(len(d))
fig, ax = plt.subplots(figsize=(10, 8))
ax.hlines(y, d.F1_strict, d.F1_lenient, color="lightgrey", lw=3, zorder=1)
for m, mk in [("strict", "o"), ("partial", "."), ("lenient", "o")]:
    lo, hi = ci[m].loc[d.index, "lo"], ci[m].loc[d.index, "hi"]
    if m != "partial":
        ax.errorbar(d[f"F1_{m}"], y, xerr=[d[f"F1_{m}"] - lo, hi - d[f"F1_{m}"]],
                    fmt="none", ecolor=MCOLOR[m], alpha=0.4, capsize=2)
    ax.scatter(d[f"F1_{m}"], y, color=MCOLOR[m], marker=mk, s=60 if m != "partial" else 80,
               zorder=3, label=m)
x_txt = np.maximum(ci["lenient"].loc[d.index, "hi"].values, ci["strict"].loc[d.index, "hi"].values)
for yi, (s, l, n, xt) in enumerate(zip(d.F1_strict, d.F1_lenient, d.n_gold, x_txt)):
    ax.text(xt + 0.02, yi, f"+{l - s:.2f}   (n={n})", va="center", fontsize=8)
ax.set_yticks(y, [short(l) for l in d.index])
for t, l in zip(ax.get_yticklabels(), d.index):
    t.set_color(FCOLOR[LABEL2FRAME[l]])
for m in ["strict", "lenient"]:
    ax.axvline(overall[f"F1_{m}"].iloc[0], color=MCOLOR[m], ls="--", lw=1,
               label=f"micro {m} = {overall[f'F1_{m}'].iloc[0]:.3f}")
ax.set_xlim(0, 1.15)
ax.set_xlabel("F1")
ax.set_title("F1 per label: strict vs. lenient")
ax.grid(axis="y", visible=False)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
save(fig, "01_strict_vs_lenient_labels")

### Per frame
Micro P, R and F1 per frame: solid bars are strict, hatched bars lenient. The hatched part is what is recovered by fixing boundaries only.

In [ ]:
d = pd.concat([per_frame, overall.rename(index={"<micro>": "ALL"})])
x = np.arange(len(d))
w = 0.26
fig, ax = plt.subplots(figsize=(12, 5.5))
for k, (met, col) in enumerate([("P", "#4C72B0"), ("R", "#DD8452"), ("F1", "#55A868")]):
    xs = x + (k - 1) * w
    ax.bar(xs, d[f"{met}_lenient"], w, color="none", edgecolor=col, hatch="///", lw=1)
    ax.bar(xs, d[f"{met}_strict"], w, color=col, label=f"{met} strict")
    for xi, v in zip(xs, d[f"{met}_lenient"]):
        ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=7, rotation=90)
ax.bar(0, 0, color="none", edgecolor="k", hatch="///", label="lenient gain")
ax.set_xticks(x, d.index, rotation=20, ha="right")
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Strict vs. lenient per frame (micro)")
ax.grid(axis="x", visible=False)
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=4, frameon=False)
save(fig, "02_strict_vs_lenient_frames")

### Precision and recall shift
Each arrow goes from the strict to the lenient point of a label: long arrows indicate boundary errors, short ones missed or spurious spans.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8))
r_ = np.linspace(0.001, 1, 600)
for f in np.arange(0.1, 1.0, 0.1):
    pp = f * r_ / (2 * r_ - f)
    msk = (r_ > f / 2) & (pp > 0) & (pp <= 1)
    ax.plot(r_[msk], pp[msk], c="grey", lw=0.5, alpha=0.4)
for lab, row in per_label.iterrows():
    col = FCOLOR[row.frame]
    ax.annotate("", xy=(row.R_lenient, row.P_lenient), xytext=(row.R_strict, row.P_strict),
                arrowprops=dict(arrowstyle="->", color=col, lw=1.2))
    ax.scatter(row.R_strict, row.P_strict, color=col, s=25, zorder=3)
    ax.annotate(short(lab), (row.R_lenient, row.P_lenient), fontsize=7,
                xytext=(4, 3), textcoords="offset points")
ax.plot([0, 1], [0, 1], "k:", lw=0.8)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02); ax.set_aspect("equal")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("From strict (dot) to lenient (arrow)")
ax.legend(handles=frame_legend, fontsize=7, loc="lower right")
save(fig, "03_pr_shift")

### Error breakdown per label
Left: what happens to gold spans. Right: share of predictions that overlap no gold span.

In [ ]:
cat_colors = ["#55A868", "#DD8452", "#C44E52", "#8172B2", "#4d4d4d"]
d = breakdown[CATEGORIES].iloc[::-1]
frac = d.div(d.sum(axis=1), axis=0)
y = np.arange(len(d))
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14, 8), sharey=True, gridspec_kw={"width_ratios": [3, 1]})
left = np.zeros(len(d))
for c, col in zip(CATEGORIES, cat_colors):
    ax.barh(y, frac[c], left=left, color=col, label=c)
    for yi, (l, v) in enumerate(zip(left, frac[c])):
        if v > 0.06:
            ax.text(l + v / 2, yi, f"{v:.0%}", ha="center", va="center", fontsize=7, color="white")
    left += frac[c]
ax.set_yticks(y, [short(l) for l in d.index])
for t, l in zip(ax.get_yticklabels(), d.index):
    t.set_color(FCOLOR[LABEL2FRAME[l]])
ax.set_xlim(0, 1)
ax.set_xlabel("Share of gold spans")
ax.set_title("Gold spans by outcome")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=5, frameon=False, fontsize=8)
ax.grid(False)

sp = (breakdown["spurious (pred)"] / per_label.n_pred.replace(0, np.nan)).reindex(d.index).fillna(0)
ax2.barh(y, sp, color="#C44E52")
for yi, v in enumerate(sp):
    ax2.text(v + 0.01, yi, f"{v:.0%}", va="center", fontsize=7)
ax2.set_xlim(0, 1)
ax2.set_xlabel("Share of predictions")
ax2.set_title("Spurious predictions")
ax2.grid(False)
save(fig, "04_error_breakdown")

### Label confusion within frames
Rows are gold labels, columns the label of the closest prediction (or none), normalised by row: the diagonal is the share found with the right label, even with imprecise boundaries.

In [ ]:
multi = [f for f in FRAMES if gold.loc[gold.frame == f, "label"].nunique() > 1]
ncol = 3
nrow = int(np.ceil(len(multi) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 4.6 * nrow))
for ax, fr in zip(axes.flat, multi):
    g = gold[gold.frame == fr]
    labs = [l for l in LABELS if LABEL2FRAME[l] == fr]
    cm = pd.crosstab(g.label, g.label_pred).reindex(index=labs, columns=labs + ["(none)"], fill_value=0)
    cmn = cm.div(cm.sum(axis=1), axis=0)
    ax.imshow(cmn.values, cmap="Blues", vmin=0, vmax=1)
    for i in range(cmn.shape[0]):
        for j in range(cmn.shape[1]):
            v = cmn.values[i, j]
            ax.text(j, i, f"{v:.2f}\n({cm.values[i, j]})", ha="center", va="center", fontsize=7,
                    color="white" if v > 0.55 else "black")
    strip = lambda s: short(s).split("_", 1)[-1] if s != "(none)" else s
    ax.set_xticks(range(cmn.shape[1]), [strip(c) for c in cmn.columns], rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(cmn.shape[0]), [strip(c) for c in cmn.index], fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("gold")
    ax.set_title(fr, color=FCOLOR[fr])
    ax.grid(False)
for ax in list(axes.flat)[len(multi):]:
    ax.axis("off")
save(fig, "05_label_confusion")

## Boundary error examples
A sample of lenient but non-exact matches, by type (too long, too short, shifted). Set `LABEL` to inspect a single label.

In [ ]:
print("Boundary errors by label and type:")
display(pd.crosstab(boundary_examples.label, boundary_examples.type).reindex(LABELS).fillna(0).astype(int))

LABEL = None          # e.g. "COOKING_CREATION_Food_Material"; None = all labels
N_EXAMPLES = 20
e = boundary_examples if LABEL is None else boundary_examples[boundary_examples.label == LABEL]
e.sample(min(N_EXAMPLES, len(e)), random_state=SEED)[["label", "type", "gold", "pred"]]

## Save

In [ ]:
breakdown.to_csv(OUT_DIR / "error_breakdown.tsv", sep="\t")
boundary_examples.to_csv(OUT_DIR / "boundary_errors.tsv", sep="\t", index=False)
pd.concat({m: ci[m] for m in MODES}, axis=1).to_csv(OUT_DIR / "bootstrap_ci_f1.tsv", sep="\t")
print("Saved to", OUT_DIR)
for f in sorted(OUT_DIR.iterdir()):
    print("  ", f.name)